# Lab 1 — Deploy a model and call it

**Required · 45 minutes · Level 100**

## What will you do?

You will deploy a language model in Microsoft Foundry, then call it from Python and shape the answers it gives you.

A **model** is trained software that generates text. You cannot call a model in the catalog directly. You first create a **deployment**: your own named, running copy of that model inside your project. From then on, your code only ever names the deployment.

Think of the catalog as a library shelf and the deployment as the copy you borrowed. Your code cites the copy you borrowed, not the shelf.

```text
model catalog  ->  your deployment  ->  responses.create(model=..., input=...)  ->  answer
```

In this lab you will:

1. Deploy an approved model in the Foundry portal.
2. Connect to your project from Python.
3. Send your first request and read what comes back.
4. Separate the standing rules from the user's question.
5. Compare the two answers.

One thing to settle now: deploying a model does **not** train it. Your deployment knows nothing about your workshop, your documents or your company. Lab 3 solves that problem.

## New words

- **Model** — the trained engine that generates text, for example `gpt-5.6-luna`.
- **Deployment** — a named instance of a model inside your project. Your code calls this name, not the catalog name.
- **Project endpoint** — the HTTPS address of your Foundry project. It identifies the project; it is not a secret and not a password.
- **Prompt** — the text you send. It steers one request. It never changes the model.
- **Responses API** — the request shape you use to call a deployment: `client.responses.create(...)`.
- **Token** — the unit a model reads and writes, roughly three quarters of a word. Tokens are also the unit you are billed in.

## Before you start

- Python 3.11 or later, with a notebook kernel selected in the top-right corner.
- Azure CLI. Run `az login` in a terminal before you begin.
- An Azure subscription with a Foundry project, and permission to deploy a model into it.

**How the To-Do sections work.** Each To-Do states a goal, gives you the steps, and offers a hint. In the code, a blank looks like `...` and carries a `# TODO` comment. Replace the `...`, then run the cell with **Shift+Enter**. If you leave a blank open, the cell stops and names it instead of sending a broken request to Azure. Open **Show solution code** after you have tried, not before.

Never paste an access key or token into this notebook. Authentication goes through `az login`.

In [ ]:
%pip install -q "azure-ai-projects==2.3.0" "azure-identity==1.25.3" "openai==2.54.0"

## 0. Connect to your project

The next cell does the housekeeping:

- reads two nonsecret settings, your project endpoint and your deployment name,
- signs in with the Azure CLI login you already have, so no key is stored here,
- builds the client you use for every request in this lab.

It also defines `check_todos`, a small helper. It is not part of the exercise: it just stops a cell with a readable message when a `...` blank is still open.

**Run the cell. You should see** `Ready to call: <your deployment name>`.

If it raises `Set AZURE_AI_PROJECT_ENDPOINT ...`, your two settings are missing. Fill them in the cell and run it again. Nothing has been sent to Azure yet.

In [ ]:
import os
import sys

from azure.ai.projects import AIProjectClient
from azure.identity import AzureCliCredential

# Your two nonsecret settings. Paste them here or set them as environment variables.
PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT", "")
MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "")

if sys.version_info < (3, 11):
    raise RuntimeError(
        "These notebooks need Python 3.11 or later. This kernel is "
        f"{sys.version_info.major}.{sys.version_info.minor}. Select a newer kernel."
    )

if not PROJECT_ENDPOINT or not MODEL_DEPLOYMENT:
    raise ValueError(
        "Set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME, "
        "or paste your own values into the two lines above."
    )


def check_todos(**answers: object) -> None:
    """Helper. Stops the cell while a `...` blank is still open."""
    still_open = [name for name, value in answers.items() if value is ...]
    if still_open:
        raise ValueError(f"Fill in these blanks first: {', '.join(still_open)}")


credential = AzureCliCredential()
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
client = project.get_openai_client(timeout=60, max_retries=0)
print(f"Ready to call: {MODEL_DEPLOYMENT}")

## 1. Deploy the model

A deployment binds four choices together. Only one of them ever appears in your code, and that surprises most people the first time.

| Choice | What it controls | Where your code sees it |
|---|---|---|
| Model and version | Which engine answers you | Nowhere. It is fixed when you deploy. |
| Deployment name | The name you call | `model="<your deployment name>"` |
| Deployment type | Where it runs and how you pay for it | Nowhere. It shows up on the bill. |
| Capacity, in tokens per minute | How much traffic you get before throttling | As `429` errors when you exceed it |

Deployment type is worth a second look, because it is where cost and isolation are decided:

![Deployment options in Microsoft Foundry, from standard shared capacity through to provisioned throughput](https://learn.microsoft.com/en-us/azure/foundry/concepts/media/deployments-overview/deployment-options-hierarchy.png)

*Deployment options hierarchy. Source: [Deployment overview](https://learn.microsoft.com/en-us/azure/foundry/concepts/deployments-overview) on Microsoft Learn.*

Start with a standard deployment. It is the cheapest way to get working, and you can move to reserved capacity later without changing a line of application code. In production, this row is a cost decision more than a technical one.

That is the practical reason the distinction matters. You can move from one model version to a newer one by changing the deployment, and no application code changes. Hard-code a catalog name instead and you lose that.

### To-Do 1 — Deploy a model

**Goal:** a deployment showing **Succeeded** in the portal, whose name you can read back.

**Steps**

1. Open [Microsoft Foundry](https://ai.azure.com/) and select your project.
2. Go to **Models + endpoints**, select **Deploy model**, and choose a model that supports the Responses API.
3. Keep the default version and deployment type, and leave capacity at its default. This lab sends only a handful of requests.
4. Give the deployment a name you will recognise, then deploy. Wait for the status to reach **Succeeded**.
5. Open the deployment in the playground and send one short message. Confirm you get an answer.
6. Copy the **deployment name** into the setup cell above, then run that cell again. Copy the deployment name, not the catalog model name. They are often similar and rarely identical.

**Predict:** your service enforces a rate limit of 100 requests per minute, a detail written down only in your own internal documentation. Your fresh deployment has just come online. Will it know that limit? Say why before you continue.

<details><summary>Answer to the prediction</summary>

No. The model was trained long before your documentation existed, and nothing you did during deployment gave it new facts. Deployment is provisioning, not learning. Asked directly, it will either say it does not know or invent a plausible-sounding number — which is worse, because a number looks like an answer. Lab 3 gives a model real sources to work from.

</details>

## 2. Send your first request

Every call in this lab uses the same method, `client.responses.create(...)`. Three parameters carry everything that matters:

| Parameter | Carries | How often it changes |
|---|---|---|
| `model` | Your deployment name | Almost never |
| `instructions` | Who the assistant is and the rules it follows | Almost never. It is your application's contract. |
| `input` | What this user asked, right now | Every single request |

Start with the smallest useful call: just `model` and `input`. You will add `instructions` in the next section and see what it buys you.

What comes back is a response object, not a string. Three parts of it are worth knowing:

- `response.status` — `"completed"` when the model finished. Any other value means you should not trust the text.
- `response.output_text` — the answer, already joined into one string for you.
- `response.usage` — the tokens this request cost you, split into input and output.

### To-Do 2 — Complete the request

**Goal:** one completed response from your own deployment.

**Steps**

1. Set `REQUEST_MODEL` to the deployment you want to call.
2. Set `REQUEST_INPUT` to the text you want to send.
3. Run the cell. Read the status, then the token count, then the answer.

Both values already exist in this notebook as variables. You do not need to type any name by hand.

**Run the cell. You should see** `status: completed`, a token count, and a short explanation of what an AI agent is.

<details><summary>Hint</summary>

Assign the variables themselves, without quotation marks. `REQUEST_MODEL = "MODEL_DEPLOYMENT"` sends the literal text `MODEL_DEPLOYMENT` to Azure as a deployment name, and Azure returns a `404`.

</details>

<details><summary>Show solution code</summary>

```python
REQUEST_MODEL = MODEL_DEPLOYMENT
REQUEST_INPUT = QUESTION
```

</details>

In [ ]:
QUESTION = "What is an AI agent? Answer in two sentences."

REQUEST_MODEL = ...  # TODO 2: name the deployment to call.
REQUEST_INPUT = ...  # TODO 2: supply the text to send.
check_todos(REQUEST_MODEL=REQUEST_MODEL, REQUEST_INPUT=REQUEST_INPUT)

plain = client.responses.create(model=REQUEST_MODEL, input=REQUEST_INPUT)

print("status:", plain.status)
print("tokens:", plain.usage.input_tokens, "in,", plain.usage.output_tokens, "out")
print()
print(plain.output_text)

## 3. Separate the rules from the question

Your first call worked, but it mixed two different things into one string. `"What is an AI agent? Answer in two sentences."` contains a question from a user *and* a formatting rule from you, the developer. Users never write the second half.

Real applications keep them apart:

```text
instructions  ->  written once by you, applies to every user      "You explain AI to beginners..."
input         ->  written by the user, different every time       "What is an AI agent?"
```

Splitting them buys you three things. You can change the assistant's behaviour without touching the code that handles user messages. You can log and review the rules on their own. And you can reuse one set of rules across many questions.

Good instructions answer three questions. A vague instruction produces a vague answer, so be specific:

| Your instructions should state | Weak version | Stronger version |
|---|---|---|
| Who you are writing for | "Be clear." | "The reader has never used AI before." |
| What the answer should look like | "Keep it short." | "Use at most three bullet points." |
| What to do when you do not know | Nothing at all | "Say you are not sure rather than guessing." |

That last row matters more than it looks. A model with no instruction about uncertainty will fill the gap with something fluent and wrong.

### To-Do 3 — Write the standing instructions

**Goal:** a second answer to the same question that visibly follows rules you wrote.

**Steps**

1. Write `INSTRUCTIONS` as one quoted string covering all three rows of the table above.
2. Predict how the answer will change. The deployment and the question stay exactly the same.
3. Run the cell and compare the two answers side by side.

**Run the cell. You should see** the original answer, then a second answer that matches the audience and the format you asked for.

<details><summary>Hint</summary>

Write it as one string in three sentences: who the reader is, what shape the answer takes, and what to do when unsure. Do not repeat the question itself — that belongs in `input`.

</details>

<details><summary>Show solution code</summary>

```python
INSTRUCTIONS = (
    "You explain AI concepts to people who have never used AI before. "
    "Answer in at most three short bullet points, in plain language. "
    "If you are not sure about something, say so instead of guessing."
)
```

Other wordings work. What matters is that a reader of your instructions can predict the shape of the answer.

</details>

In [ ]:
INSTRUCTIONS = ...  # TODO 3: write the standing rules for this assistant.
check_todos(INSTRUCTIONS=INSTRUCTIONS)

shaped = client.responses.create(
    model=MODEL_DEPLOYMENT,
    instructions=INSTRUCTIONS,
    input=QUESTION,
)

print("BEFORE, no instructions\n")
print(plain.output_text)
print("\n" + "-" * 60 + "\n")
print("AFTER, with your instructions\n")
print(shaped.output_text)

## Deterministic success check

Model wording changes every time, so this check does not look for particular words. It confirms that both requests completed, that both returned text, and that your instructions actually changed the result.

In [ ]:
assert plain.status == "completed", "The first request did not complete."
assert shaped.status == "completed", "The instructed request did not complete."
assert plain.output_text.strip(), "The first response came back empty."
assert shaped.output_text.strip(), "The instructed response came back empty."
assert plain.output_text != shaped.output_text, (
    "Both answers are identical. Check that INSTRUCTIONS is a non-empty string "
    "and that you ran the cell above after filling it in."
)
print("PASS - one deployment answered twice, and your instructions changed the second answer.")

## What you learned

- A **model** is the engine; a **deployment** is the named instance your code calls. Swapping the deployment behind a stable name is how teams upgrade models without shipping code.
- The Responses API takes three things that matter: `model`, `instructions` and `input`.
- **Instructions** are your standing contract with the model. **Input** is one user's message. Keeping them apart is what separates a prototype from an application.
- Deploying a model gives it no new knowledge. It has no access to your documents, your systems or today's date.

**Reflection.** Answer these in one sentence each before you close the notebook.

1. Someone hard-codes the catalog model name into ten services. What breaks when the team upgrades the model?
2. Which of your three instruction rules did the model follow most visibly? Which did you not get to test?
3. Your assistant now needs to answer "what is the rate limit on our orders API?" What is still missing?

<details><summary>Compare your answers</summary>

1. Every one of the ten services needs a code change and a redeployment. With a deployment name, one portal change covers all ten.
2. Format is usually the most visible. The uncertainty rule stays untested until you ask something the model cannot know — which is exactly what Lab 3 does.
3. A source of truth. The model has never seen your internal documentation, and no instruction can create facts it never had. Lab 3 connects it to a searchable set of documents.

</details>

**If something fails:** check the kernel selected in the top-right corner, the account behind `az login`, the spelling of the deployment name, and whether the chosen model supports the Responses API. A `403` means your signed-in account lacks a role on the project. A `429` means you exceeded the capacity you deployed — wait, then retry.

**Expected artifact:** two answers from one deployment you can identify, and a passing success check.

**Next:** Lab 2 moves the instructions out of your Python file and into Foundry, where they become a versioned agent.

In [ ]:
client.close()
project.close()
credential.close()
print("Closed the local clients. Your Azure deployment is untouched.")